# Phase 3 exploration: typewell-derived TVT features

**Goal:** decide whether GR-template matching against the typewell can produce a
useful per-row TVT estimate for LGBM v2. If so, port to `src/rogii_wellbore/features.py`
and use as features (`gr_match_tvt`, `gr_match_sim`, `gr_match_delta_anchor`).

**Status (in-progress):** the signal exists strongly in the data, but two matcher
designs aimed at per-row TVT extraction failed (38–41 RMSE vs CF 23 on the
exploration well). The conclusion below proposes pivoting to a *per-well TVT
trend* feature instead.

**Method:** explore one well end-to-end, then sanity-check the core assumption
(GR signal is informative) across a 30-well stratified sample.

> This notebook is the *exploration* scratchpad. The eventual Phase 3 deliverable
> notebook is `03_typewell.ipynb` (separate). All exploration commits feed into
> the design decisions documented at the bottom.

## Setup

One train well, not in the public-LB test set. Loaded from the parquet cache.

In [ ]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rogii_wellbore.data import list_wells, load_horizontal, load_typewell
from rogii_wellbore.features import _gr_interpolate, pick_inference_anchor

TEST_WELLS = {"000d7d20", "00bbac68", "00e12e8b"}

# Pick first train well that's not in the test set.
all_train = list_wells("train")
WELL_ID = next(w for w in all_train if w not in TEST_WELLS)
print(f"Exploring well: {WELL_ID}")

In [ ]:
well = load_horizontal("train", well_ids=[WELL_ID], source="parquet").reset_index(drop=True)
tw = load_typewell("train", well_ids=[WELL_ID], source="parquet").reset_index(drop=True)

print("well cols:", list(well.columns))
print("tw cols:  ", list(tw.columns))
print(f"\nwell rows: {len(well)}, tw rows: {len(tw)}")
print("\ntypewell head:")
print(tw.head())

tw_tvt_raw = tw["TVT"].to_numpy(dtype=float)
tw_gr_raw = tw["GR"].to_numpy(dtype=float)
print(f"\ntw TVT range: {tw_tvt_raw.min():.1f} to {tw_tvt_raw.max():.1f}")
print(f"tw native step (first 5 diffs): {np.diff(tw_tvt_raw)[:5]}")

# Resample typewell to 1.0 ft TVT grid (lateral MD step is 1.0 ft; matching grids = clean cross-correlation)
order = np.argsort(tw_tvt_raw)
tw_tvt_sorted, tw_gr_sorted = tw_tvt_raw[order], tw_gr_raw[order]

# Handle any NaN in typewell GR before interpolating onto the grid
finite = np.isfinite(tw_gr_sorted)
if not finite.all():
    print(f"WARNING: typewell has {(~finite).sum()} NaN GR rows; dropping before resample")
    tw_tvt_sorted, tw_gr_sorted = tw_tvt_sorted[finite], tw_gr_sorted[finite]

tvt_grid = np.arange(np.ceil(tw_tvt_sorted.min()), np.floor(tw_tvt_sorted.max()) + 1.0, 1.0)
gr_grid = np.interp(tvt_grid, tw_tvt_sorted, tw_gr_sorted)
print(f"\nresampled typewell: {len(tvt_grid)} rows, TVT {tvt_grid[0]:.1f} to {tvt_grid[-1]:.1f}")

## Attempt 1: row-space matcher (MVP)

**Hypothesis:** for each lateral row, slide a fixed-row-width GR window across
the typewell within a ±radius of the anchor TVT; argmax Pearson correlation
gives the predicted TVT. Centered vs causal window as a knob.

Below: slow-but-correct MVP, then both window shapes run on the one well.

In [ ]:
def gr_match_mvp(
    lateral_gr: np.ndarray,
    typewell_tvt: np.ndarray,
    typewell_gr: np.ndarray,
    anchor_tvt: float,
    window_len: int = 101,
    radius: float = 75.0,
    window_shape: str = "centered",  # "centered" or "causal"
) -> tuple[np.ndarray, np.ndarray]:
    """Slow, correct MVP. Returns (gr_match_tvt, gr_match_sim), len = len(lateral_gr)."""
    n = len(lateral_gr)
    out_tvt = np.full(n, np.nan)
    out_sim = np.full(n, np.nan)

    if window_shape == "centered":
        left = window_len // 2
        right = window_len - left
    elif window_shape == "causal":
        left, right = window_len - 1, 1
    else:
        raise ValueError(window_shape)

    cand_j = np.flatnonzero(np.abs(typewell_tvt - anchor_tvt) <= radius)
    if len(cand_j) == 0:
        return out_tvt, out_sim

    for i in range(n):
        lo, hi = i - left, i + right
        if lo < 0 or hi > n:
            continue
        lat = lateral_gr[lo:hi]
        if np.isnan(lat).any():
            continue
        a = lat - lat.mean()
        a_norm = np.sqrt((a * a).sum())
        if a_norm == 0:
            continue

        best_sim, best_tvt = -np.inf, np.nan
        for j in cand_j:
            tj_lo, tj_hi = j - left, j + right
            if tj_lo < 0 or tj_hi > len(typewell_gr):
                continue
            tw_w = typewell_gr[tj_lo:tj_hi]
            b = tw_w - tw_w.mean()
            b_norm = np.sqrt((b * b).sum())
            if b_norm == 0:
                continue
            sim = (a * b).sum() / (a_norm * b_norm)
            if sim > best_sim:
                best_sim, best_tvt = sim, typewell_tvt[j]

        if np.isfinite(best_sim):
            out_tvt[i] = best_tvt
            out_sim[i] = best_sim

    return out_tvt, out_sim

In [ ]:
lat_gr = _gr_interpolate(well["GR"].to_numpy(dtype=float))
anchor_idx = pick_inference_anchor(well)
anchor_tvt = well["TVT_input"].to_numpy(dtype=float)[anchor_idx]
print(
    f"anchor_idx={anchor_idx}, anchor_tvt={anchor_tvt:.2f}, "
    f"n_rows={len(well)}, eval_rows={len(well) - anchor_idx - 1}"
)

t0 = time.time()
tvt_c, sim_c = gr_match_mvp(lat_gr, tvt_grid, gr_grid, anchor_tvt, window_shape="centered")
print(f"centered: {time.time() - t0:.1f}s")

t0 = time.time()
tvt_z, sim_z = gr_match_mvp(lat_gr, tvt_grid, gr_grid, anchor_tvt, window_shape="causal")
print(f"causal:   {time.time() - t0:.1f}s")

In [ ]:
true_tvt = well["TVT"].to_numpy(dtype=float)
row = np.arange(len(well))
eval_mask = np.isnan(well["TVT_input"].to_numpy(dtype=float))

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})

ax = axes[0]
ax.plot(row, true_tvt, color="black", lw=1.2, label="true TVT")
ax.plot(row[eval_mask], tvt_c[eval_mask], ".", ms=2.5, alpha=0.7, label="gr_match (centered)")
ax.plot(row[eval_mask], tvt_z[eval_mask], ".", ms=2.5, alpha=0.7, label="gr_match (causal)")
ax.axvline(anchor_idx, color="red", ls="--", alpha=0.5, label="anchor")
ax.axhline(anchor_tvt, color="red", ls=":", alpha=0.3)
ax.set_ylabel("TVT")
ax.set_title(f"{WELL_ID}: gr_match prediction vs true TVT (eval zone only)")
ax.legend(loc="best")

ax = axes[1]
ax.plot(row[eval_mask], sim_c[eval_mask], ".", ms=2.5, alpha=0.7, label="centered")
ax.plot(row[eval_mask], sim_z[eval_mask], ".", ms=2.5, alpha=0.7, label="causal")
ax.set_ylabel("gr_match_sim")
ax.set_xlabel("row_idx")
ax.set_ylim(-0.2, 1.05)
ax.legend(loc="best")

plt.tight_layout()
plt.savefig("../reports/figures/06_gr_match_eval.png", dpi=100, bbox_inches="tight")
plt.show()

# Quick numeric summary on eval rows
for name, pred in [("centered", tvt_c), ("causal", tvt_z)]:
    m = eval_mask & np.isfinite(pred)
    rmse = np.sqrt(((pred[m] - true_tvt[m]) ** 2).mean()) if m.any() else float("nan")
    cov = m.sum() / eval_mask.sum()
    print(f"{name:8s} eval RMSE = {rmse:.3f}   coverage = {cov:.1%}")

### Result: matcher is much worse than carry-forward

- centered eval RMSE = 56.5 (cov 99%)
- causal   eval RMSE = 48.6 (cov 100%)
- (carry-forward baseline = ~23 on this well)

Predictions cluster in two horizontal bands around the anchor, not tracking truth.
Sim scores stay low (0.2–0.6).

Plot saved: `reports/figures/06_gr_match_eval.png`.

### Diagnostic: are GR signals comparable?

Before chasing matcher mechanics, confirm the lateral and typewell GR look like
they could match at all — same dynamic range, distinctive structure within the
search radius.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6))

# Top: lateral GR through full well
ax = axes[0]
ax.plot(row, lat_gr, lw=0.6, color="steelblue")
ax.axvline(anchor_idx, color="red", ls="--", alpha=0.5, label="anchor")
ax.set_ylabel("lateral GR")
ax.set_title(f"{WELL_ID}: lateral GR (full well)")
ax.legend()

# Bottom: typewell GR through the search range, with true-eval-TVT range overlaid
ax = axes[1]
mask_search = np.abs(tvt_grid - anchor_tvt) <= 75
ax.plot(tvt_grid[mask_search], gr_grid[mask_search], lw=0.8, color="darkorange")
ax.axvline(anchor_tvt, color="red", ls="--", alpha=0.5, label="anchor TVT")
true_eval_tvt = true_tvt[eval_mask]
ax.axvspan(
    np.nanmin(true_eval_tvt),
    np.nanmax(true_eval_tvt),
    color="green",
    alpha=0.15,
    label=f"true eval TVT range [{np.nanmin(true_eval_tvt):.0f}, {np.nanmax(true_eval_tvt):.0f}]",
)
ax.set_xlabel("typewell TVT")
ax.set_ylabel("typewell GR")
ax.set_title(f"typewell GR across search range (\u00b175 ft of anchor {anchor_tvt:.1f})")
ax.legend()

plt.tight_layout()
plt.savefig("../reports/figures/07_lateral_gr.png", dpi=100, bbox_inches="tight")
plt.show()

# Also: lateral GR scale vs typewell GR scale — are they even comparable?
print(
    f"lateral GR: mean={np.nanmean(lat_gr):.1f}, std={np.nanstd(lat_gr):.1f}, "
    f"range=[{np.nanmin(lat_gr):.1f}, {np.nanmax(lat_gr):.1f}]"
)
print(
    f"typewell GR (search range): mean={gr_grid[mask_search].mean():.1f}, std={gr_grid[mask_search].std():.1f}, "
    f"range=[{gr_grid[mask_search].min():.1f}, {gr_grid[mask_search].max():.1f}]"
)

**Reading:** lateral GR is rich (mean 89, std 19, distinctive bumps), typewell
GR in the search range is comparable (mean 88, std 15, varied shape). Scales
match, content looks matchable. So the matcher *failure* isn't "no signal" — it's
a design problem. But before rebuilding the matcher: confirm the signal *can* be
recovered when we have ground truth.

## The acid test

In the **known zone** we have true `TVT_input`. So we can plot lateral GR vs
*true* TVT and overlay the typewell GR vs TVT. If they line up, GR-matching is
fundamentally viable — and the matcher's job becomes "do it without true TVT in
the eval zone."

If they don't line up even with true TVT, no matcher can save us.

In [ ]:
# Acid test: in the KNOWN zone we have true TVT. Does lateral GR(TVT) match typewell GR(TVT)?
known_mask = ~np.isnan(well["TVT_input"].to_numpy(dtype=float))
lat_gr_full = _gr_interpolate(well["GR"].to_numpy(dtype=float))
lat_tvt_known = well["TVT_input"].to_numpy(dtype=float)[known_mask]
lat_gr_known = lat_gr_full[known_mask]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(
    tw_tvt_sorted, tw_gr_sorted, color="darkorange", lw=0.8, alpha=0.8, label="typewell GR(TVT)"
)
ax.plot(
    lat_tvt_known,
    lat_gr_known,
    color="steelblue",
    lw=0.8,
    alpha=0.8,
    label="lateral GR vs true TVT (known zone)",
)
ax.set_xlabel("TVT")
ax.set_ylabel("GR")
ax.set_title(f"{WELL_ID}: do lateral & typewell GR agree where TVT is KNOWN?")
ax.set_xlim(lat_tvt_known.min() - 20, lat_tvt_known.max() + 20)
ax.legend()
plt.tight_layout()
fig.savefig("../reports/figures/08_acid_test_known_zone.png", dpi=100, bbox_inches="tight")
plt.show()

print(
    f"known-zone TVT span: {lat_tvt_known.min():.0f} to {lat_tvt_known.max():.0f} "
    f"({lat_tvt_known.max() - lat_tvt_known.min():.0f} ft)"
)
# Correlation where they overlap, on a common 1ft TVT grid
lo = max(lat_tvt_known.min(), tw_tvt_sorted.min())
hi = min(lat_tvt_known.max(), tw_tvt_sorted.max())
grid = np.arange(np.ceil(lo), np.floor(hi) + 1, 1.0)
order = np.argsort(lat_tvt_known)
lat_on_grid = np.interp(grid, lat_tvt_known[order], lat_gr_known[order])
tw_on_grid = np.interp(grid, tw_tvt_sorted, tw_gr_sorted)
r = np.corrcoef(lat_on_grid, tw_on_grid)[0, 1]
print(f"overlap span: {len(grid)} ft;  Pearson r (lateral vs typewell on true TVT) = {r:.3f}")

### Result: **r = 0.879** ✅

The lateral and typewell GR signatures track each other bump-for-bump across the
494-ft known-zone TVT span. Big spikes line up at TVT 11750 and 12120. Broad
lows around 12050 line up. The dip-and-recover texture in the middle aligns.

**This proves the GR signal *is* informative for TVT recovery on this well.**
The previous matcher's failure must be a geometry/window design problem, not a
"signal doesn't exist" problem. Worth iterating.

Plot saved: `reports/figures/08_acid_test_known_zone.png`.

## Picking a TVT proxy for the eval zone

The matcher needs a TVT search center for each eval row. Two candidates:

- **Z-derived:** `proxy_tvt = anchor_tvt + (Z - anchor_Z)` — assumes `dTVT ≈ dZ`.
- **Constant:** `proxy_tvt = anchor_tvt` (carry-forward).

We can evaluate both directly because we have true TVT on this train well.

In [ ]:
from rogii_wellbore.features import _gr_interpolate, pick_inference_anchor

z = well["Z"].to_numpy(dtype=float)
tvt_input = well["TVT_input"].to_numpy(dtype=float)
true_tvt = well["TVT"].to_numpy(dtype=float)
lat_gr = _gr_interpolate(well["GR"].to_numpy(dtype=float))

anchor_idx = pick_inference_anchor(well)
anchor_tvt = tvt_input[anchor_idx]
anchor_z = z[anchor_idx]
eval_mask = np.isnan(tvt_input)
known_mask = ~eval_mask

# z-derived proxy: assume dTVT ~= dZ (flat-formation assumption)
proxy_tvt = anchor_tvt + (z - anchor_z)

resid_known = tvt_input[known_mask] - proxy_tvt[known_mask]
print("KNOWN-zone, z-derived proxy (anchor_tvt + dZ):")
print(
    f"  resid (true-proxy): mean={resid_known.mean():+.2f} std={resid_known.std():.2f} "
    f"min={resid_known.min():+.1f} max={resid_known.max():+.1f}"
)

resid_eval = true_tvt[eval_mask] - proxy_tvt[eval_mask]
print("EVAL-zone, z-derived proxy (cheating w/ true TVT, train well only):")
print(
    f"  resid (true-proxy): mean={resid_eval.mean():+.2f} std={resid_eval.std():.2f} "
    f"min={resid_eval.min():+.1f} max={resid_eval.max():+.1f}"
)

resid_const = true_tvt[eval_mask] - anchor_tvt
print("EVAL-zone, constant proxy (anchor_tvt = carry-forward):")
print(
    f"  resid (true-anchor): mean={resid_const.mean():+.2f} std={resid_const.std():.2f} "
    f"min={resid_const.min():+.1f} max={resid_const.max():+.1f}"
)

### Result: constant proxy wins decisively

| proxy | known-zone resid std | eval-zone resid std | eval max |resid| |
|---|---|---|---|
| z-derived (anchor + dZ) | **282 ft** 🚫 | 61 | 200.9 |
| constant (anchor_tvt) | 0 by construction | **12.1** | 37.8 |

**`dZ` is a terrible proxy for `dTVT`.** TVT is thickness in the *formation* frame
(accounts for structural dip); Z is true vertical depth in the *earth* frame. They
diverge whenever beds dip. In the build zone, dZ overshoots TVT change by 1000 ft.
In the eval zone, dZ undershoots by mean +106 ft.

This is also why Phase 2's `linear_extrap` baseline scored RMSE 107.77 — the
trajectory just doesn't predict TVT.

**Decisions:**
- Matcher proxy = `anchor_tvt` (constant). The matcher's job is to find the
  per-row *correction* from this flat baseline.
- Search radius **R = 50 ft** comfortably covers the eval-zone residual range
  (max |resid| = 37.8 on this well).

## Attempt 2: TVT-space matcher (constant proxy)

Same row-window on the lateral side, but matching happens in TVT space on the
typewell side — for each row, slide a typewell window of equal length across
±50 ft of anchor and snap to argmax correlation.

In [ ]:
def gr_match_tvt_space(
    lat_gr: np.ndarray,  # interpolated lateral GR, full well
    tvt_grid: np.ndarray,  # typewell TVT, 1ft grid, ascending
    gr_grid: np.ndarray,  # typewell GR on that grid
    anchor_tvt: float,
    w_ft: float = 25.0,  # window half-width in TVT feet
    R_ft: float = 50.0,  # search radius around anchor (proxy)
    proxy_tvt: np.ndarray | None = None,  # per-row proxy; None => constant anchor
) -> tuple[np.ndarray, np.ndarray]:
    """For each row, slide a w_ft typewell window across [proxy-R, proxy+R],
    return the TVT of best Pearson correlation against the lateral GR window.

    The lateral GR window for row i is the w_ft of typewell-grid-spaced lateral
    GR centered on row i (in ROW space — we use a fixed row window as a local
    GR-shape probe, since we lack per-row true TVT to resample on).
    Matching happens in TVT space on the typewell side.
    """
    n = len(lat_gr)
    out_tvt = np.full(n, np.nan)
    out_sim = np.full(n, np.nan)

    win = int(round(2 * w_ft))  # row window length (~50 rows for w_ft=25)
    half = win // 2
    step = float(np.median(np.diff(tvt_grid)))  # ~1.0
    tw_win = int(round(2 * w_ft / step))  # typewell window length in grid pts
    tw_half = tw_win // 2

    for i in range(n):
        lo, hi = i - half, i + half
        if lo < 0 or hi > n:
            continue
        a = lat_gr[lo:hi]
        if np.isnan(a).any():
            continue
        a = a - a.mean()
        an = np.sqrt((a * a).sum())
        if an == 0:
            continue

        center = proxy_tvt[i] if proxy_tvt is not None else anchor_tvt
        j_cand = np.flatnonzero(np.abs(tvt_grid - center) <= R_ft)
        best_sim, best_tvt = -np.inf, np.nan
        for j in j_cand:
            jl, jh = j - tw_half, j - tw_half + len(a)
            if jl < 0 or jh > len(gr_grid):
                continue
            b = gr_grid[jl:jh]
            if len(b) != len(a):
                continue
            b = b - b.mean()
            bn = np.sqrt((b * b).sum())
            if bn == 0:
                continue
            sim = (a * b).sum() / (an * bn)
            if sim > best_sim:
                best_sim, best_tvt = sim, tvt_grid[j]
        if np.isfinite(best_sim):
            out_tvt[i], out_sim[i] = best_tvt, best_sim

    return out_tvt, out_sim

In [ ]:
import time

t0 = time.time()
gm_tvt, gm_sim = gr_match_tvt_space(lat_gr, tvt_grid, gr_grid, anchor_tvt, w_ft=25.0, R_ft=50.0)
print(f"matcher: {time.time() - t0:.1f}s")

m = eval_mask & np.isfinite(gm_tvt)
rmse_match = np.sqrt(((gm_tvt[m] - true_tvt[m]) ** 2).mean())
rmse_cf = np.sqrt(((anchor_tvt - true_tvt[eval_mask]) ** 2).mean())
print(f"eval coverage: {m.sum() / eval_mask.sum():.1%}")
print(f"gr_match  eval RMSE = {rmse_match:.2f}")
print(f"carry-fwd eval RMSE = {rmse_cf:.2f}   <- the bar to beat")

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
row = np.arange(len(well))
axes[0].plot(row, true_tvt, "k", lw=1.2, label="true TVT")
axes[0].plot(row[m], gm_tvt[m], ".", ms=2.5, alpha=0.6, label="gr_match")
axes[0].axhline(anchor_tvt, color="red", ls=":", alpha=0.5, label="anchor (carry-fwd)")
axes[0].axvline(anchor_idx, color="red", ls="--", alpha=0.4)
axes[0].set_ylabel("TVT")
axes[0].legend()
axes[0].set_title(
    f"{WELL_ID}: TVT-space matcher  (match RMSE {rmse_match:.1f} vs CF {rmse_cf:.1f})"
)
axes[1].plot(row[m], gm_sim[m], ".", ms=2.5, alpha=0.6)
axes[1].set_ylabel("gr_match_sim")
axes[1].set_xlabel("row_idx")
axes[1].set_ylim(-0.2, 1.05)
plt.tight_layout()
fig.savefig("../reports/figures/09_tvt_matcher.png", dpi=100, bbox_inches="tight")
plt.show()

### Result: still bad — RMSE 41.5 vs CF 23.2

Coverage 99.4%, runtime fine, but predictions cluster in **two horizontal bands**
(~12190 and ~12260) — the matcher is repeatedly snapping to two specific TVTs
across thousands of different lateral rows. Sim scores 0.3–0.6, far below the
known-zone r=0.88.

**Diagnosis:** for the matcher to pick the same TVT for many different lateral
rows, the lateral GR windows must look similar — i.e. the lateral window isn't
carrying local TVT information. The eval zone is **near-horizontal in TVT** (37
ft of TVT spread over 4000+ rows). 50 rows of lateral MD ≈ 1–3 ft of TVT. So
we're comparing 50 rows of *within-bed lateral heterogeneity* against 50 ft of
*vertical stratigraphy*. The geometry doesn't match.

Plot saved: `reports/figures/09_tvt_matcher.png`.

**Before iterating again: is this well representative? Or did we get unlucky on
a flat well?**

## Acid test across 30 wells

If the acid-test r=0.88 is well-specific, GR-matching might be fundamentally
unstable across the field. If it holds widely, the signal is real and we should
keep iterating on the matcher.

**Sample:** 30 wells, stratified across eval-zone length, GR NaN rate, and
typewell coverage (test wells excluded).

In [ ]:
# Load per-well stats from EDA cache
eval_stats = pd.read_parquet("../data/interim/eval_zone_stats.parquet")
gr_stats = pd.read_parquet("../data/interim/gr_per_well_stats.parquet")
tw_cov = pd.read_parquet("../data/interim/typewell_coverage.parquet")

# Quick inspection of columns so we pick the right ones
print("eval_stats:", list(eval_stats.columns))
print("gr_stats:  ", list(gr_stats.columns))
print("tw_cov:    ", list(tw_cov.columns))
print(f"\nrows: eval_stats={len(eval_stats)}, gr_stats={len(gr_stats)}, tw_cov={len(tw_cov)}")
print("\neval_stats head:")
print(eval_stats.head(3))

In [ ]:
TEST_WELLS = {"000d7d20", "00bbac68", "00e12e8b"}
RNG = np.random.default_rng(42)

# Join everything we need on well_id index
stats = (
    eval_stats[["eval_frac", "n_eval", "n_known"]]
    .join(gr_stats[["nan_rate"]], how="inner")
    .join(tw_cov[["below_gap", "above_gap", "covers_full"]], how="inner")
)
stats = stats.drop(index=[w for w in TEST_WELLS if w in stats.index])
print(f"Pool: {len(stats)} train wells (test wells excluded)")

# Strata
short_eval = stats.nsmallest(int(0.20 * len(stats)), "eval_frac").index
long_eval = stats.nlargest(int(0.20 * len(stats)), "eval_frac").index
high_nan = stats[stats["nan_rate"] > 0.50].index
below_cov = stats[stats["below_gap"] > 0].index
rest = (
    stats.index.difference(short_eval)
    .difference(long_eval)
    .difference(high_nan)
    .difference(below_cov)
)


def sample(idx, k, label):
    k = min(k, len(idx))
    picks = RNG.choice(np.asarray(idx), size=k, replace=False)
    print(f"  {label:18s} pool={len(idx):4d}, picked={k}")
    return list(picks)


picks = []
print("Stratified picks:")
picks += sample(short_eval, 6, "short eval")
picks += sample(long_eval, 6, "long eval")
picks += sample(high_nan, 6, "high GR NaN")
picks += sample(below_cov, 6, "below tw-cov")
picks += sample(rest, 6, "rest")

sample_wells = sorted(set(picks))
print(f"\nUnique wells in sample: {len(sample_wells)}")
print("First 5:", sample_wells[:5])

In [ ]:
from rogii_wellbore.data import load_horizontal, load_typewell
from rogii_wellbore.features import _gr_interpolate


def acid_test_one(well_id: str) -> dict:
    """Return r and metadata for one well's known-zone acid test."""
    try:
        w = load_horizontal("train", well_ids=[well_id], source="parquet").reset_index(drop=True)
        tw = load_typewell("train", well_ids=[well_id], source="parquet").reset_index(drop=True)

        # Typewell prep (sorted, no-NaN GR)
        tt = tw["TVT"].to_numpy(dtype=float)
        tg = tw["GR"].to_numpy(dtype=float)
        order = np.argsort(tt)
        tt, tg = tt[order], tg[order]
        finite = np.isfinite(tg)
        tt, tg = tt[finite], tg[finite]

        # Lateral known-zone GR(TVT)
        known = ~np.isnan(w["TVT_input"].to_numpy(dtype=float))
        lat_gr = _gr_interpolate(w["GR"].to_numpy(dtype=float))[known]
        lat_tvt = w["TVT_input"].to_numpy(dtype=float)[known]
        order = np.argsort(lat_tvt)
        lat_tvt, lat_gr = lat_tvt[order], lat_gr[order]

        # Overlap on common 1ft grid
        lo = max(lat_tvt.min(), tt.min())
        hi = min(lat_tvt.max(), tt.max())
        if hi - lo < 50:  # not enough overlap to be meaningful
            return {
                "well_id": well_id,
                "r": np.nan,
                "overlap_ft": hi - lo,
                "known_tvt_span": lat_tvt.max() - lat_tvt.min(),
            }
        grid = np.arange(np.ceil(lo), np.floor(hi) + 1, 1.0)
        lat_on = np.interp(grid, lat_tvt, lat_gr)
        tw_on = np.interp(grid, tt, tg)
        r = np.corrcoef(lat_on, tw_on)[0, 1]
        return {
            "well_id": well_id,
            "r": float(r),
            "overlap_ft": float(hi - lo),
            "known_tvt_span": float(lat_tvt.max() - lat_tvt.min()),
        }
    except Exception as e:
        return {
            "well_id": well_id,
            "r": np.nan,
            "overlap_ft": np.nan,
            "known_tvt_span": np.nan,
            "error": str(e),
        }


results = pd.DataFrame([acid_test_one(w) for w in sample_wells]).set_index("well_id")
# Join in the strata info for analysis
results = results.join(stats, how="left")
print(
    results[
        ["r", "overlap_ft", "known_tvt_span", "eval_frac", "nan_rate", "below_gap"]
    ].sort_values("r", ascending=False)
)

In [ ]:
print("\n--- Summary of known-zone r across sample ---")
print(results["r"].describe().round(3))
print(f"\nFraction with r > 0.7: {(results['r'] > 0.7).mean():.0%}")
print(f"Fraction with r > 0.5: {(results['r'] > 0.5).mean():.0%}")
print(f"Fraction with r < 0.3: {(results['r'] < 0.3).mean():.0%}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(results["r"].dropna(), bins=20, edgecolor="black", alpha=0.8)
ax.axvline(0.879, color="green", ls="--", alpha=0.6, label="015fe0d2 (0.88)")
ax.axvline(0.7, color="black", ls=":", alpha=0.4)
ax.set_xlabel("Pearson r (lateral GR vs typewell GR on true TVT, known zone)")
ax.set_ylabel("wells")
ax.set_title(f"Acid test across {len(results)} stratified wells")
ax.legend()
plt.tight_layout()
fig.savefig("../reports/figures/10_acid_test_sample.png", dpi=100, bbox_inches="tight")
plt.show()

### Result: **GR-matching is broadly viable** ✅

- mean r = 0.84, median 0.83, **min 0.66, max 0.95**
- **100%** of wells have r > 0.5
- **93%** of wells have r > 0.7
- Below-typewell-coverage wells (5 in sample) all score r in 0.79–0.83 — coverage
  gaps aren't a failure mode where overlap exists
- High GR NaN wells (60–70% NaN) all score r > 0.79 — linear interpolation
  preserves enough signal
- 015fe0d2 (the exploration well, r=0.88) sits at the 75th percentile — *above*
  average. We were not lucky-outlier; most wells will be slightly harder.

**Conclusion:** the previous matchers' failures are *matcher-design* problems,
not signal-availability problems. Keep iterating, but with awareness that the
per-row geometry challenge is real.

Plot saved: `reports/figures/10_acid_test_sample.png`.

## Attempt 3: stretched matcher (TVT-aware window sizing)

Direct attack on the geometry problem: the lateral window's *TVT span* (estimated
from |dZ| across the window, accepting the dZ ≠ dTVT bias as a scale rather than
position issue) determines a resampled lateral window length. The typewell window
matches that length. So we're comparing lateral GR shape across ~25 ft of TVT
extent against typewell GR shape across ~25 ft of TVT extent.

Defaults: MD window half-width 200 rows, search radius ±50 ft of anchor, min TVT
span 10 ft (below which the lateral is too horizontal to carry usable shape).

In [ ]:
def gr_match_stretched(
    lat_gr: np.ndarray,
    lat_z: np.ndarray,
    tvt_grid: np.ndarray,  # typewell, 1ft ascending
    gr_grid: np.ndarray,  # typewell GR
    anchor_tvt: float,
    W_rows: int = 200,  # MD window half-width (rows)
    R_ft: float = 50.0,  # search radius around anchor
    min_tvt_span: float = 10.0,  # below this, return NaN
) -> tuple[np.ndarray, np.ndarray]:
    """For each row, estimate the lateral window's TVT span from dZ, resample
    the lateral GR onto a 1ft-equivalent TVT grid of that span, slide across
    typewell within +/- R_ft of anchor, return best TVT + Pearson r."""
    n = len(lat_gr)
    out_tvt = np.full(n, np.nan)
    out_sim = np.full(n, np.nan)

    # Pre-compute typewell candidate indices (constant per call)
    j_lo = np.searchsorted(tvt_grid, anchor_tvt - R_ft)
    j_hi = np.searchsorted(tvt_grid, anchor_tvt + R_ft)
    cand_idx = np.arange(j_lo, j_hi)

    for i in range(W_rows, n - W_rows):
        lat_window = lat_gr[i - W_rows : i + W_rows + 1]
        if np.isnan(lat_window).any():
            continue

        # Estimate TVT span of this window from dZ
        tvt_span_est = abs(lat_z[i + W_rows] - lat_z[i - W_rows])
        if tvt_span_est < min_tvt_span:
            continue

        # Resample lateral GR onto a 1ft-equivalent grid of length L
        L = int(round(tvt_span_est))
        if L < 5:
            continue
        x_src = np.linspace(0.0, 1.0, len(lat_window))
        x_tgt = np.linspace(0.0, 1.0, L)
        lat_resampled = np.interp(x_tgt, x_src, lat_window)
        a = lat_resampled - lat_resampled.mean()
        an = np.sqrt((a * a).sum())
        if an == 0:
            continue

        half_L = L // 2
        best_sim, best_tvt = -np.inf, np.nan
        for j in cand_idx:
            jl, jh = j - half_L, j - half_L + L
            if jl < 0 or jh > len(gr_grid):
                continue
            b = gr_grid[jl:jh]
            if len(b) != L:
                continue
            b = b - b.mean()
            bn = np.sqrt((b * b).sum())
            if bn == 0:
                continue
            sim = (a * b).sum() / (an * bn)
            if sim > best_sim:
                best_sim, best_tvt = sim, tvt_grid[j]

        if np.isfinite(best_sim):
            out_tvt[i], out_sim[i] = best_tvt, best_sim

    return out_tvt, out_sim

In [ ]:
lat_z = well["Z"].to_numpy(dtype=float)

t0 = time.time()
gm_tvt2, gm_sim2 = gr_match_stretched(
    lat_gr,
    lat_z,
    tvt_grid,
    gr_grid,
    anchor_tvt,
    W_rows=200,
    R_ft=50.0,
    min_tvt_span=10.0,
)
print(f"stretched matcher: {time.time() - t0:.1f}s")

m = eval_mask & np.isfinite(gm_tvt2)
rmse_match = np.sqrt(((gm_tvt2[m] - true_tvt[m]) ** 2).mean()) if m.any() else float("nan")
rmse_cf = np.sqrt(((anchor_tvt - true_tvt[eval_mask]) ** 2).mean())
cov = m.sum() / eval_mask.sum()
print(f"coverage:  {cov:.1%}")
print(f"stretched: eval RMSE = {rmse_match:.2f}")
print("yesterday: eval RMSE = 41.50  (row-window TVT-space matcher)")
print(f"carry-fwd: eval RMSE = {rmse_cf:.2f}   <- the bar")

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
row = np.arange(len(well))
axes[0].plot(row, true_tvt, "k", lw=1.2, label="true TVT")
axes[0].plot(row[m], gm_tvt2[m], ".", ms=2.5, alpha=0.6, label="gr_match (stretched)")
axes[0].axhline(anchor_tvt, color="red", ls=":", alpha=0.5, label="anchor")
axes[0].axvline(anchor_idx, color="red", ls="--", alpha=0.4)
axes[0].set_ylabel("TVT")
axes[0].set_title(
    f"{WELL_ID}: stretched matcher  (match {rmse_match:.1f} vs CF {rmse_cf:.1f}, cov {cov:.0%})"
)
axes[0].legend()
axes[1].plot(row[m], gm_sim2[m], ".", ms=2.5, alpha=0.6)
axes[1].set_ylabel("gr_match_sim")
axes[1].set_xlabel("row_idx")
axes[1].set_ylim(-0.2, 1.05)
plt.tight_layout()
fig.savefig("../reports/figures/11_stretched_matcher.png", dpi=100, bbox_inches="tight")
plt.show()

### Result: still bad — RMSE 38.0 vs CF 23.2

Slightly better than attempts 1 and 2 (41.5 → 38.0), coverage now 83% (the
~200-row edge padding + the min-TVT-span cutoff), sim scores meaningfully higher
(0.5–0.9 in clusters) — but **still worse than CF**.

The improvement direction is right (sim is better, predictions less degenerate),
but the per-row geometry challenge defeats the matcher.

Plot saved: `reports/figures/11_stretched_matcher.png`.

## Conclusions — and the proposed pivot

### What we learned

1. **The GR signal is real and broadly available.** Acid test across 30 wells
   shows median r = 0.83 in the known zone, 100% of wells above 0.5. GR-matching
   against the typewell is fundamentally viable as a TVT-recovery technique.

2. **Z is not a TVT proxy.** `dZ ≠ dTVT` because TVT is in the formation frame
   (accounts for structural dip) while Z is true vertical depth. The flat
   constant proxy (`anchor_tvt`) is the only safe one for the matcher's search
   center.

3. **Per-row TVT prediction fails in flat eval zones.** The lateral is
   near-horizontal in TVT through the eval zone (~37 ft over 4000 rows on
   015fe0d2). A row-window on the lateral side simply doesn't sample enough TVT
   extent to disambiguate among nearby typewell positions. Three matcher
   variants (row-space, TVT-space with constant proxy, stretched) all hit 38–56
   RMSE vs CF's 23.

4. **The geometry, not the algorithm, is the wall.** The signal carries TVT
   information only when the lateral *moves* in TVT. The known zone has 494 ft
   of TVT to traverse — that's why r=0.88 works there. The eval zone has 30–40
   ft — there's not enough per-row TVT resolution in the data, regardless of
   matcher cleverness.

### Pivot: per-well TVT trend, not per-row TVT

The eval-zone signal that *does* exist at full strength is the bulk GR sequence
across thousands of horizontal rows. That signal can answer a coarser but more
tractable question:

> Is the lateral drifting up, down, or staying flat across the eval zone,
> and by how much?

**Proposed Attempt 4:** for each well, find the single best (TVT offset, TVT
slope) such that `gr_match_tvt[i] = anchor + offset + slope * (i - anchor_idx)`
maximizes the global Pearson correlation between the entire eval-zone lateral
GR sequence and the typewell GR at the corresponding TVTs. Use the full
eval-zone signal (long horizontal traverse → strong statistical power) to fit
two parameters instead of one TVT per row.

This matches the resolution the data actually carries: low-frequency drift
information, not per-row TVT.

### Phase 3 implication

The per-row matcher approach is shelved. Idea-A is the new lead. If it beats
CF on this well, expand to the 30-well sweep, then build into
`src/rogii_wellbore/features.py` as `gr_trend_offset`, `gr_trend_slope`,
`gr_trend_sim` (3 features per row, constant within a well — LGBM will need to
interact them with `dmd` to extract drift trend).

If Attempt 4 also fails, fallback options are (B) known-zone calibration carried
into the eval zone, or (C) GR-context statistical features without an explicit
matcher.

### Next session

Implement Attempt 4 on this well first. If RMSE < CF (23.2), expand to sweep.
If not, look at the residual structure and choose between (B) and (C).

In [ ]:
# --- Attempt 4 prep: how good can a 2-param (offset, slope) model possibly be?
# Fit (offset, slope) directly against TRUE TVT on this well's eval zone.
# This is the CEILING - matcher can't do better than this regardless of design.

eval_idx = np.flatnonzero(eval_mask)
true_eval = true_tvt[eval_mask]

# Design: TVT_pred[i] = anchor_tvt + offset + slope * (i - anchor_idx)
# So: (true_TVT - anchor_tvt) = offset + slope * (i - anchor_idx)
# Linear regression in (i - anchor_idx) on (true - anchor).
x = (eval_idx - anchor_idx).astype(float)
y = true_eval - anchor_tvt

slope_opt, offset_opt = np.polyfit(x, y, 1)  # returns [slope, intercept]
fitted = anchor_tvt + offset_opt + slope_opt * x

rmse_ceiling = np.sqrt(((fitted - true_eval) ** 2).mean())
rmse_cf = np.sqrt(((anchor_tvt - true_eval) ** 2).mean())

print("CEILING (cheating fit against true TVT):")
print(f"  offset = {offset_opt:+.2f} ft")
print(
    f"  slope  = {slope_opt:+.5f} ft/row  (= {slope_opt * len(x):+.1f} ft total drift across eval)"
)
print(f"  rmse   = {rmse_ceiling:.2f}")
print("")
print(f"CF baseline:        {rmse_cf:.2f}")
print("Yesterday stretched: 38.04")
print("")
print(f"Ceiling improvement over CF: {rmse_cf - rmse_ceiling:+.2f} RMSE")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
row_full = np.arange(len(well))
ax.plot(row_full, true_tvt, "k", lw=1.5, label="true TVT")
ax.plot(
    eval_idx, fitted, "r-", lw=1.5, alpha=0.8, label=f"affine ceiling  (RMSE {rmse_ceiling:.1f})"
)
ax.axhline(
    anchor_tvt, color="gray", ls=":", alpha=0.6, label=f"carry-forward  (RMSE {rmse_cf:.1f})"
)
ax.axvline(anchor_idx, color="red", ls="--", alpha=0.4, label="anchor")
ax.set_xlabel("row_idx")
ax.set_ylabel("TVT")
ax.set_title(f"{WELL_ID}: how much of TVT drift can a 2-param affine model capture?")
ax.legend()
plt.tight_layout()
fig.savefig("../reports/figures/12_affine_ceiling.png", dpi=100, bbox_inches="tight")
plt.show()

# Residual structure - is what's left after the line just noise, or systematic?
resid = true_eval - fitted
print(
    f"\nresidual after affine fit:  mean={resid.mean():+.3f} std={resid.std():.2f} "
    f"min={resid.min():+.1f} max={resid.max():+.1f}"
)

In [ ]:
def affine_ceiling_one(well_id: str) -> dict:
    """Fit a 2-param affine model (offset + slope) against TRUE TVT on the
    eval zone. Returns CF and ceiling RMSE for this well."""
    try:
        w = load_horizontal("train", well_ids=[well_id], source="parquet").reset_index(drop=True)
        tvt_input = w["TVT_input"].to_numpy(dtype=float)
        true_tvt = w["TVT"].to_numpy(dtype=float)

        eval_mask = np.isnan(tvt_input)
        known_idx = np.flatnonzero(~eval_mask)
        if len(known_idx) == 0 or not eval_mask.any():
            return {"well_id": well_id, "error": "no eval or no known"}
        anchor_idx = int(known_idx[-1])
        anchor_tvt = float(tvt_input[anchor_idx])

        eval_idx = np.flatnonzero(eval_mask)
        true_eval = true_tvt[eval_mask]

        # Skip if true TVT has NaNs in eval (shouldn't happen on train wells)
        if np.isnan(true_eval).any():
            return {"well_id": well_id, "error": "true_tvt has nan in eval"}

        x = (eval_idx - anchor_idx).astype(float)
        y = true_eval - anchor_tvt
        slope, offset = np.polyfit(x, y, 1)
        fitted = anchor_tvt + offset + slope * x

        rmse_ceiling = float(np.sqrt(((fitted - true_eval) ** 2).mean()))
        rmse_cf = float(np.sqrt(((anchor_tvt - true_eval) ** 2).mean()))

        return {
            "well_id": well_id,
            "rmse_cf": rmse_cf,
            "rmse_ceiling": rmse_ceiling,
            "improvement": rmse_cf - rmse_ceiling,
            "offset": float(offset),
            "slope": float(slope),
            "total_drift": float(slope * len(x)),
            "eval_n": int(len(x)),
        }
    except Exception as e:
        return {"well_id": well_id, "error": str(e)}


ceiling_results = pd.DataFrame([affine_ceiling_one(w) for w in sample_wells]).set_index("well_id")
print(ceiling_results.sort_values("improvement", ascending=False).round(2).to_string())

In [ ]:
ok = ceiling_results.dropna(subset=["rmse_cf", "rmse_ceiling"]).copy()
print(f"\nWells successfully fit: {len(ok)}/{len(ceiling_results)}")
print("\n--- Distributions ---")
print(ok[["rmse_cf", "rmse_ceiling", "improvement", "total_drift"]].describe().round(2))

print(f"\nWells where ceiling beats CF:        {(ok['improvement'] > 0).sum()}/{len(ok)}")
print(f"Wells where ceiling beats CF by >2:  {(ok['improvement'] > 2).sum()}/{len(ok)}")
print(f"Wells where ceiling LOSES to CF:     {(ok['improvement'] < 0).sum()}/{len(ok)}")

# Pooled-style RMSE comparison (weighted by eval_n, matching the competition metric)
total_sse_cf = (ok["rmse_cf"] ** 2 * ok["eval_n"]).sum()
total_sse_ceiling = (ok["rmse_ceiling"] ** 2 * ok["eval_n"]).sum()
total_n = ok["eval_n"].sum()
pooled_cf = np.sqrt(total_sse_cf / total_n)
pooled_ceiling = np.sqrt(total_sse_ceiling / total_n)
print(f"\nPooled RMSE across {len(ok)} wells:")
print(f"  CF:      {pooled_cf:.2f}")
print(f"  Ceiling: {pooled_ceiling:.2f}")
print(f"  Delta:   {pooled_cf - pooled_ceiling:+.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(ok["rmse_cf"], ok["rmse_ceiling"], s=40, alpha=0.7)
mx = max(ok["rmse_cf"].max(), ok["rmse_ceiling"].max()) * 1.05
ax.plot([0, mx], [0, mx], "k--", alpha=0.4, label="y = x (no improvement)")
ax.set_xlabel("CF RMSE per well")
ax.set_ylabel("affine-ceiling RMSE per well")
ax.set_title("Ceiling vs CF (below line = ceiling wins)")
ax.legend()
ax.set_xlim(0, mx)
ax.set_ylim(0, mx)

ax = axes[1]
ax.hist(ok["improvement"], bins=20, edgecolor="black", alpha=0.8)
ax.axvline(0, color="red", ls="--", alpha=0.6, label="no improvement")
ax.set_xlabel("CF RMSE - ceiling RMSE  (>0 = ceiling wins)")
ax.set_ylabel("wells")
ax.set_title(f"Affine ceiling improvement over CF ({len(ok)} wells)")
ax.legend()

plt.tight_layout()
fig.savefig("../reports/figures/13_affine_ceiling_sweep.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
def gr_trend_match(
    lat_gr: np.ndarray,  # full-well lateral GR (already interpolated)
    eval_mask: np.ndarray,  # bool, True on eval rows
    anchor_idx: int,
    anchor_tvt: float,
    tvt_grid: np.ndarray,  # typewell TVT, 1ft ascending
    gr_grid: np.ndarray,  # typewell GR on tvt_grid
    R_offset: float = 50.0,  # offset search radius (ft)
    R_slope: float = 50.0,  # slope*eval_len search radius (ft) -- "total drift"
    n_offset: int = 101,
    n_slope: int = 101,
) -> dict:
    """Fit (offset, slope) by max Pearson r between lateral GR on the eval zone
    and typewell GR sampled at predicted_tvt[i] = anchor + offset + slope*(i-anchor_idx).

    Returns dict with offset, slope, sim (max Pearson r), and a full-length
    predicted_tvt array (NaN outside eval zone).
    """
    n = len(lat_gr)
    eval_idx = np.flatnonzero(eval_mask)
    if len(eval_idx) < 50:
        return {
            "offset": np.nan,
            "slope": np.nan,
            "sim": np.nan,
            "predicted_tvt": np.full(n, np.nan),
        }

    x = (eval_idx - anchor_idx).astype(float)  # row offsets from anchor
    lat_eval = lat_gr[eval_idx]
    if np.isnan(lat_eval).any():
        # Should not happen post-interpolation, but defensive.
        return {
            "offset": np.nan,
            "slope": np.nan,
            "sim": np.nan,
            "predicted_tvt": np.full(n, np.nan),
        }

    # Demean lateral once for correlation
    lat_dm = lat_eval - lat_eval.mean()
    lat_norm = np.sqrt((lat_dm * lat_dm).sum())
    if lat_norm == 0:
        return {
            "offset": np.nan,
            "slope": np.nan,
            "sim": np.nan,
            "predicted_tvt": np.full(n, np.nan),
        }

    # Search grids
    offsets = np.linspace(-R_offset, R_offset, n_offset)  # ft
    eval_len = len(eval_idx)
    slope_max = R_slope / max(eval_len, 1)  # ft/row
    slopes = np.linspace(-slope_max, slope_max, n_slope)

    best_sim = -np.inf
    best_offset = np.nan
    best_slope = np.nan

    tw_lo, tw_hi = tvt_grid[0], tvt_grid[-1]

    for off in offsets:
        # tvt at each eval row for slope=0 baseline (precompute, broadcast later)
        base_tvt = anchor_tvt + off
        for slp in slopes:
            tvt_pred = base_tvt + slp * x
            # Out-of-typewell-coverage rows: np.interp clamps to ends; we want
            # to instead exclude them from the correlation so they don't dominate.
            in_cov = (tvt_pred >= tw_lo) & (tvt_pred <= tw_hi)
            if in_cov.sum() < 50:
                continue
            tw_at = np.interp(tvt_pred[in_cov], tvt_grid, gr_grid)
            lat_w = lat_eval[in_cov]
            a = lat_w - lat_w.mean()
            b = tw_at - tw_at.mean()
            an = np.sqrt((a * a).sum())
            bn = np.sqrt((b * b).sum())
            if an == 0 or bn == 0:
                continue
            sim = (a * b).sum() / (an * bn)
            if sim > best_sim:
                best_sim = sim
                best_offset = float(off)
                best_slope = float(slp)

    # Build full-well predicted_tvt
    predicted_tvt = np.full(n, np.nan)
    if np.isfinite(best_sim):
        all_x = (np.arange(n) - anchor_idx).astype(float)
        predicted_tvt[eval_mask] = anchor_tvt + best_offset + best_slope * all_x[eval_mask]

    return {
        "offset": best_offset,
        "slope": best_slope,
        "sim": float(best_sim) if np.isfinite(best_sim) else np.nan,
        "predicted_tvt": predicted_tvt,
    }

In [ ]:
import time

t0 = time.time()
res = gr_trend_match(
    lat_gr,
    eval_mask,
    anchor_idx,
    anchor_tvt,
    tvt_grid,
    gr_grid,
    R_offset=50.0,
    R_slope=50.0,
    n_offset=101,
    n_slope=101,
)
elapsed = time.time() - t0
print(f"matcher: {elapsed:.1f}s   ({101 * 101} grid points)")

# Compare to known ceiling (Cell 17 set offset_opt, slope_opt, rmse_ceiling)
print("\n               offset       slope    total_drift    RMSE     sim")
print(
    f"ceiling (cheat) {offset_opt:+7.2f}  {slope_opt:+9.5f}    "
    f"{slope_opt * len(eval_idx):+7.1f}      {rmse_ceiling:5.2f}     n/a"
)
print(
    f"matcher         {res['offset']:+7.2f}  {res['slope']:+9.5f}    "
    f"{res['slope'] * len(eval_idx):+7.1f}      ",
    end="",
)

m = eval_mask & np.isfinite(res["predicted_tvt"])
rmse_matcher = float(np.sqrt(((res["predicted_tvt"][m] - true_tvt[m]) ** 2).mean()))
print(f"{rmse_matcher:5.2f}    {res['sim']:.3f}")
print(f"CF baseline:                                          {rmse_cf:5.2f}")
print(f"\nmatcher vs CF:      {rmse_cf - rmse_matcher:+.2f} RMSE")
print(f"matcher vs ceiling: {rmse_matcher - rmse_ceiling:+.2f} RMSE (gap to optimum)")

# Plot
fig, ax = plt.subplots(figsize=(13, 5))
row_full = np.arange(len(well))
ax.plot(row_full, true_tvt, "k", lw=1.5, label="true TVT")
ax.plot(
    eval_idx,
    fitted,
    color="green",
    lw=1.5,
    alpha=0.7,
    label=f"affine ceiling  (RMSE {rmse_ceiling:.1f})",
)
ax.plot(
    row_full[m],
    res["predicted_tvt"][m],
    color="red",
    lw=1.5,
    alpha=0.7,
    label=f"matcher  (RMSE {rmse_matcher:.1f}, sim {res['sim']:.2f})",
)
ax.axhline(
    anchor_tvt, color="gray", ls=":", alpha=0.6, label=f"carry-forward  (RMSE {rmse_cf:.1f})"
)
ax.axvline(anchor_idx, color="red", ls="--", alpha=0.4)
ax.set_xlabel("row_idx")
ax.set_ylabel("TVT")
ax.set_title(f"{WELL_ID}: gr_trend_match vs ceiling vs CF")
ax.legend()
plt.tight_layout()
fig.savefig("../reports/figures/14_gr_trend_matcher.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
def _sim_at(
    offset, slope, lat_gr, eval_idx, anchor_idx, anchor_tvt, tvt_grid, gr_grid, smooth=None
):
    """Pearson r at a given (offset, slope), optionally with rolling-mean smoothing."""
    x = (eval_idx - anchor_idx).astype(float)
    tvt_pred = anchor_tvt + offset + slope * x
    tw_lo, tw_hi = tvt_grid[0], tvt_grid[-1]
    in_cov = (tvt_pred >= tw_lo) & (tvt_pred <= tw_hi)
    if in_cov.sum() < 50:
        return np.nan
    tw_at = np.interp(tvt_pred[in_cov], tvt_grid, gr_grid)
    lat_w = lat_gr[eval_idx][in_cov]
    if smooth is not None and smooth > 1:
        s = pd.Series
        lat_w = s(lat_w).rolling(smooth, min_periods=smooth // 2, center=True).mean().to_numpy()
        tw_at = s(tw_at).rolling(smooth, min_periods=smooth // 2, center=True).mean().to_numpy()
        m = np.isfinite(lat_w) & np.isfinite(tw_at)
        lat_w, tw_at = lat_w[m], tw_at[m]
        if len(lat_w) < 50:
            return np.nan
    a = lat_w - lat_w.mean()
    b = tw_at - tw_at.mean()
    an = np.sqrt((a * a).sum())
    bn = np.sqrt((b * b).sum())
    if an == 0 or bn == 0:
        return np.nan
    return float((a * b).sum() / (an * bn))


# Evaluate at ceiling vs matcher's best, no smoothing and with smoothing
print(
    f"{'point':25s} {'sim (raw)':>10s} {'sim (sm=50)':>12s} {'sim (sm=200)':>13s} {'sim (sm=500)':>13s}"
)
for name, off, slp in [
    ("ceiling (+3.83, +0.0074)", offset_opt, slope_opt),
    ("matcher (-15.0, -0.0061)", res["offset"], res["slope"]),
    ("CF      (0, 0)", 0.0, 0.0),
]:
    sims = [
        _sim_at(off, slp, lat_gr, eval_idx, anchor_idx, anchor_tvt, tvt_grid, gr_grid, smooth=s)
        for s in (None, 50, 200, 500)
    ]
    print(f"{name:25s}  {sims[0]:>9.3f}   {sims[1]:>10.3f}   {sims[2]:>11.3f}   {sims[3]:>11.3f}")

In [ ]:
def predict_B_calibration(well_id):
    """Idea B: known-zone calibration of lateral GR to typewell-at-TVT,
    then per-row TVT lookup in eval zone."""
    try:
        w = load_horizontal("train", well_ids=[well_id], source="parquet").reset_index(drop=True)
        tw = load_typewell("train", well_ids=[well_id], source="parquet").reset_index(drop=True)

        # Typewell prep
        tt = tw["TVT"].to_numpy(dtype=float)
        tg = tw["GR"].to_numpy(dtype=float)
        ok = np.isfinite(tg)
        tt, tg = tt[ok], tg[ok]
        order = np.argsort(tt)
        tt, tg = tt[order], tg[order]
        grid = np.arange(np.ceil(tt.min()), np.floor(tt.max()) + 1, 1.0)
        twg = np.interp(grid, tt, tg)

        # Well prep
        tvt_input = w["TVT_input"].to_numpy(dtype=float)
        true_tvt = w["TVT"].to_numpy(dtype=float)
        lat_gr = _gr_interpolate(w["GR"].to_numpy(dtype=float))
        eval_mask = np.isnan(tvt_input)
        known_idx = np.flatnonzero(~eval_mask)
        anchor_idx = int(known_idx[-1])
        anchor_tvt = float(tvt_input[anchor_idx])

        # Known-zone calibration: lat_gr ~ a * typewell_at_tvt + b
        # Use only rows where lateral TVT is inside typewell coverage.
        known_lat_gr = lat_gr[~eval_mask]
        known_tvt = tvt_input[~eval_mask]
        in_cov_known = (known_tvt >= grid[0]) & (known_tvt <= grid[-1])
        if in_cov_known.sum() < 50:
            return None
        tw_at_known = np.interp(known_tvt[in_cov_known], grid, twg)
        lat_at_known = known_lat_gr[in_cov_known]
        # Robust(ish) linear fit
        a, b = np.polyfit(tw_at_known, lat_at_known, 1)

        # Eval-zone: for each row, find the TVT in [anchor-50, anchor+50]
        # whose typewell GR best matches (lat_gr[i] - b)/a.
        eval_idx = np.flatnonzero(eval_mask)  # noqa: F841
        lo = max(grid[0], anchor_tvt - 50)
        hi = min(grid[-1], anchor_tvt + 50)
        m_cand = (grid >= lo) & (grid <= hi)
        cand_tvt = grid[m_cand]
        cand_gr = twg[m_cand]
        if len(cand_tvt) < 10:
            return None

        target = (lat_gr[eval_mask] - b) / a  # typewell-frame GR
        # Nearest neighbour in GR space, within the radius window.
        # For each target value, find candidate index minimizing |cand_gr - target|.
        pred_tvt = np.full(len(w), np.nan)
        # Vectorize: broadcast (n_eval, n_cand)
        diffs = np.abs(cand_gr[None, :] - target[:, None])
        best = np.argmin(diffs, axis=1)
        pred_tvt[eval_mask] = cand_tvt[best]

        rmse = float(np.sqrt(((pred_tvt[eval_mask] - true_tvt[eval_mask]) ** 2).mean()))
        return {
            "well_id": well_id,
            "rmse": rmse,
            "n_eval": int(eval_mask.sum()),
            "calib_a": float(a),
            "calib_b": float(b),
        }
    except Exception as e:
        return {"well_id": well_id, "rmse": np.nan, "error": str(e)}


def predict_C_statistical(well_id):
    """Idea C (minimal): predict TVT = anchor + scaled_typewell_slope_at_anchor * dmd_normalized.
    Tests whether a simple deterministic function of local typewell shape beats CF."""
    try:
        w = load_horizontal("train", well_ids=[well_id], source="parquet").reset_index(drop=True)
        tw = load_typewell("train", well_ids=[well_id], source="parquet").reset_index(drop=True)

        tt = tw["TVT"].to_numpy(dtype=float)
        tg = tw["GR"].to_numpy(dtype=float)
        ok = np.isfinite(tg)
        tt, tg = tt[ok], tg[ok]
        order = np.argsort(tt)
        tt, tg = tt[order], tg[order]
        grid = np.arange(np.ceil(tt.min()), np.floor(tt.max()) + 1, 1.0)
        twg = np.interp(grid, tt, tg)

        tvt_input = w["TVT_input"].to_numpy(dtype=float)
        true_tvt = w["TVT"].to_numpy(dtype=float)
        eval_mask = np.isnan(tvt_input)
        known_idx = np.flatnonzero(~eval_mask)
        anchor_idx = int(known_idx[-1])
        anchor_tvt = float(tvt_input[anchor_idx])

        # Local typewell slope around anchor (50 ft window)
        m = (grid >= anchor_tvt - 25) & (grid <= anchor_tvt + 25)
        if m.sum() < 10:
            slope = 0.0
        else:
            slope, _ = np.polyfit(grid[m], twg[m], 1)  # GR per ft

        # Compare lateral GR mean in eval zone to anchor GR
        lat_gr = _gr_interpolate(w["GR"].to_numpy(dtype=float))
        eval_gr_mean = float(np.mean(lat_gr[eval_mask]))
        anchor_gr_local = float(np.mean(lat_gr[max(0, anchor_idx - 50) : anchor_idx + 1]))
        gr_delta = eval_gr_mean - anchor_gr_local

        # If GR rose and typewell GR rises with depth => TVT increased.
        # drift_estimate (ft) = gr_delta / slope, clipped to +- 50 ft.
        if abs(slope) < 1e-3:  # noqa: SIM108
            drift = 0.0
        else:
            drift = float(np.clip(gr_delta / slope, -50, 50))

        # Spread the drift linearly across eval rows for a per-row prediction.
        eval_idx = np.flatnonzero(eval_mask)
        x = (eval_idx - anchor_idx).astype(float)
        x_norm = x / max(x.max(), 1)
        pred_eval = anchor_tvt + drift * x_norm
        pred_tvt = np.full(len(w), np.nan)
        pred_tvt[eval_mask] = pred_eval

        rmse = float(np.sqrt(((pred_tvt[eval_mask] - true_tvt[eval_mask]) ** 2).mean()))
        return {
            "well_id": well_id,
            "rmse": rmse,
            "n_eval": int(eval_mask.sum()),
            "tw_slope": float(slope),
            "gr_delta": gr_delta,
            "drift": drift,
        }
    except Exception as e:
        return {"well_id": well_id, "rmse": np.nan, "error": str(e)}


def predict_Hybrid_simgate(well_id):
    """Hybrid: run gr_trend_match, gate by sim — pred = anchor + sim * matcher_correction.
    Low-sim wells fall back to CF; high-sim wells (if any) use the matcher."""
    try:
        w = load_horizontal("train", well_ids=[well_id], source="parquet").reset_index(drop=True)
        tw = load_typewell("train", well_ids=[well_id], source="parquet").reset_index(drop=True)

        tt = tw["TVT"].to_numpy(dtype=float)
        tg = tw["GR"].to_numpy(dtype=float)
        ok = np.isfinite(tg)
        tt, tg = tt[ok], tg[ok]
        order = np.argsort(tt)
        tt, tg = tt[order], tg[order]
        grid = np.arange(np.ceil(tt.min()), np.floor(tt.max()) + 1, 1.0)
        twg = np.interp(grid, tt, tg)

        tvt_input = w["TVT_input"].to_numpy(dtype=float)
        true_tvt = w["TVT"].to_numpy(dtype=float)
        lat_gr = _gr_interpolate(w["GR"].to_numpy(dtype=float))
        eval_mask = np.isnan(tvt_input)
        known_idx = np.flatnonzero(~eval_mask)
        anchor_idx = int(known_idx[-1])
        anchor_tvt = float(tvt_input[anchor_idx])

        res = gr_trend_match(
            lat_gr,
            eval_mask,
            anchor_idx,
            anchor_tvt,
            grid,
            twg,
            R_offset=50.0,
            R_slope=50.0,
            n_offset=51,
            n_slope=51,
        )  # coarser grid for speed
        if not np.isfinite(res["sim"]):
            return {"well_id": well_id, "rmse": np.nan, "error": "no sim"}

        # Gate the matcher's correction by sim. sim in [-1,1]; clip negative to 0.
        gate = max(0.0, res["sim"])
        matcher_pred = res["predicted_tvt"]
        # Convex blend: gate * matcher + (1-gate) * anchor
        pred_tvt = np.where(eval_mask, gate * matcher_pred + (1 - gate) * anchor_tvt, np.nan)

        rmse = float(np.sqrt(((pred_tvt[eval_mask] - true_tvt[eval_mask]) ** 2).mean()))
        return {
            "well_id": well_id,
            "rmse": rmse,
            "n_eval": int(eval_mask.sum()),
            "sim": res["sim"],
            "gate": gate,
            "offset": res["offset"],
            "slope": res["slope"],
        }
    except Exception as e:
        return {"well_id": well_id, "rmse": np.nan, "error": str(e)}

In [ ]:
import time

t0 = time.time()
B_res = pd.DataFrame([predict_B_calibration(w) for w in sample_wells]).set_index("well_id")
print(f"Idea B done in {time.time() - t0:.1f}s")

t0 = time.time()
C_res = pd.DataFrame([predict_C_statistical(w) for w in sample_wells]).set_index("well_id")
print(f"Idea C done in {time.time() - t0:.1f}s")

t0 = time.time()
H_res = pd.DataFrame([predict_Hybrid_simgate(w) for w in sample_wells]).set_index("well_id")
print(f"Hybrid done in {time.time() - t0:.1f}s")

# CF + ceiling already in ceiling_results
cmp = ceiling_results[["rmse_cf", "rmse_ceiling", "eval_n"]].copy()
cmp["rmse_B"] = B_res["rmse"]
cmp["rmse_C"] = C_res["rmse"]
cmp["rmse_Hybrid"] = H_res["rmse"]


# Pooled RMSE
def pooled(col):
    ok = cmp.dropna(subset=[col, "eval_n"])
    return float(np.sqrt((ok[col] ** 2 * ok["eval_n"]).sum() / ok["eval_n"].sum()))


print("\n--- Pooled RMSE across 30 wells ---")
print(f"  CF baseline:   {pooled('rmse_cf'):6.2f}")
print(
    f"  Idea B:        {pooled('rmse_B'):6.2f}   "
    f"({'BEATS' if pooled('rmse_B') < pooled('rmse_cf') else 'LOSES TO'} CF)"
)
print(
    f"  Idea C:        {pooled('rmse_C'):6.2f}   "
    f"({'BEATS' if pooled('rmse_C') < pooled('rmse_cf') else 'LOSES TO'} CF)"
)
print(
    f"  Hybrid:        {pooled('rmse_Hybrid'):6.2f}   "
    f"({'BEATS' if pooled('rmse_Hybrid') < pooled('rmse_cf') else 'LOSES TO'} CF)"
)
print(f"  Ceiling (cheat): {pooled('rmse_ceiling'):.2f}  <- absolute best")

print("\n--- Per-well wins vs CF ---")
for col in ["rmse_B", "rmse_C", "rmse_Hybrid"]:
    wins = (cmp[col] < cmp["rmse_cf"]).sum()
    print(f"  {col:14s} beats CF on {wins}/30 wells")

print("\n--- Full per-well table (sorted by CF RMSE descending) ---")
print(cmp.sort_values("rmse_cf", ascending=False).round(2).to_string())

In [ ]:
import lightgbm as lgb


def build_feature_row(well_id):
    """Per-eval-row feature matrix for one well. Returns DataFrame with columns:
    well_id, row_idx, dmd, dz, gr_roll_mean, gr_roll_std,            (v1 features)
    tw_slope_at_anchor, calib_a, calib_b, gr_delta_eval_anchor,      (idea B/C scalars)
    matcher_sim, matcher_offset, matcher_slope,                       (hybrid scalars)
    target_residual                                                   (TVT - anchor)
    """
    try:
        w = load_horizontal("train", well_ids=[well_id], source="parquet").reset_index(drop=True)
        tw = load_typewell("train", well_ids=[well_id], source="parquet").reset_index(drop=True)

        # Typewell prep
        tt = tw["TVT"].to_numpy(float)
        tg = tw["GR"].to_numpy(float)
        ok = np.isfinite(tg)
        tt, tg = tt[ok], tg[ok]
        order = np.argsort(tt)
        tt, tg = tt[order], tg[order]
        grid = np.arange(np.ceil(tt.min()), np.floor(tt.max()) + 1, 1.0)
        twg = np.interp(grid, tt, tg)

        # Lateral
        md = w["MD"].to_numpy(float)
        z = w["Z"].to_numpy(float)
        tvt_input = w["TVT_input"].to_numpy(float)
        true_tvt = w["TVT"].to_numpy(float)
        lat_gr = _gr_interpolate(w["GR"].to_numpy(float))
        eval_mask = np.isnan(tvt_input)
        known_idx = np.flatnonzero(~eval_mask)
        anchor_idx = int(known_idx[-1])
        anchor_tvt = float(tvt_input[anchor_idx])
        anchor_z = float(z[anchor_idx])
        anchor_md = float(md[anchor_idx])

        # v1: causal rolling GR stats
        s = pd.Series(lat_gr)
        gr_mean = s.rolling(50, min_periods=1).mean().to_numpy()
        gr_std = s.rolling(50, min_periods=2).std().to_numpy()

        # Idea C scalars: tw slope at anchor + eval/anchor GR delta
        m = (grid >= anchor_tvt - 25) & (grid <= anchor_tvt + 25)
        tw_slope = float(np.polyfit(grid[m], twg[m], 1)[0]) if m.sum() >= 10 else 0.0
        eval_gr_mean = float(np.mean(lat_gr[eval_mask]))
        anchor_gr_local = float(np.mean(lat_gr[max(0, anchor_idx - 50) : anchor_idx + 1]))
        gr_delta = eval_gr_mean - anchor_gr_local

        # Idea B scalars: known-zone calibration
        known_tvt = tvt_input[~eval_mask]
        known_lat_gr = lat_gr[~eval_mask]
        in_cov_k = (known_tvt >= grid[0]) & (known_tvt <= grid[-1])
        if in_cov_k.sum() >= 50:
            tw_at_k = np.interp(known_tvt[in_cov_k], grid, twg)
            calib_a, calib_b = np.polyfit(tw_at_k, known_lat_gr[in_cov_k], 1)
        else:
            calib_a, calib_b = 1.0, 0.0

        # Hybrid scalars: gr_trend_match results (coarser grid)
        res = gr_trend_match(
            lat_gr,
            eval_mask,
            anchor_idx,
            anchor_tvt,
            grid,
            twg,
            R_offset=50,
            R_slope=50,
            n_offset=51,
            n_slope=51,
        )
        m_sim = res["sim"] if np.isfinite(res["sim"]) else 0.0
        m_offset = res["offset"] if np.isfinite(res["offset"]) else 0.0
        m_slope = res["slope"] if np.isfinite(res["slope"]) else 0.0

        # Eval rows only
        eval_idx = np.flatnonzero(eval_mask)
        feats = pd.DataFrame(
            {
                "well_id": well_id,
                "row_idx": eval_idx,
                # v1
                "dmd": md[eval_mask] - anchor_md,
                "dz": z[eval_mask] - anchor_z,
                "gr_roll_mean": gr_mean[eval_mask],
                "gr_roll_std": gr_std[eval_mask],
                # new (per-well scalars, broadcast)
                "tw_slope_at_anchor": tw_slope,
                "calib_a": float(calib_a),
                "calib_b": float(calib_b),
                "gr_delta_eval_anchor": gr_delta,
                "matcher_sim": float(m_sim),
                "matcher_offset": float(m_offset),
                "matcher_slope": float(m_slope),
                # target: residual from anchor
                "target_residual": true_tvt[eval_mask] - anchor_tvt,
            }
        )
        return feats
    except Exception as e:
        print(f"  {well_id}: {e}")
        return None


print("Building feature matrix for 30 wells...")
t0 = time.time()
parts = [build_feature_row(w) for w in sample_wells]
parts = [p for p in parts if p is not None]
fmat = pd.concat(parts, ignore_index=True)
print(f"  built {len(fmat):,} rows from {len(parts)} wells in {time.time() - t0:.1f}s")
print(f"  columns: {list(fmat.columns)}")
print(fmat.describe().round(2).T[["count", "mean", "std", "min", "max"]])

In [ ]:
# Hold out 6 wells for validation. Stratify split: include high-CF + low-CF + below-cov + high-NaN.
RNG2 = np.random.default_rng(0)
all_wells = sorted(fmat["well_id"].unique())
val_wells = sorted(RNG2.choice(all_wells, size=6, replace=False))
train_wells = [w for w in all_wells if w not in val_wells]
print(f"train wells: {len(train_wells)}, val wells: {len(val_wells)}")

FEATURES = [c for c in fmat.columns if c not in ("well_id", "row_idx", "target_residual")]
print(f"features ({len(FEATURES)}): {FEATURES}")

tr = fmat[fmat["well_id"].isin(train_wells)]
va = fmat[fmat["well_id"].isin(val_wells)]
X_tr, y_tr = tr[FEATURES].to_numpy(), tr["target_residual"].to_numpy()
X_va, y_va = va[FEATURES].to_numpy(), va["target_residual"].to_numpy()

params = dict(
    objective="regression",
    metric="rmse",
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=100,
    feature_fraction=0.9,
    bagging_fraction=0.9,
    bagging_freq=5,
    verbose=-1,
    seed=42,
)
dtrain = lgb.Dataset(X_tr, y_tr, feature_name=FEATURES)
dval = lgb.Dataset(X_va, y_va, feature_name=FEATURES, reference=dtrain)
model = lgb.train(
    params,
    dtrain,
    num_boost_round=500,
    valid_sets=[dtrain, dval],
    valid_names=["tr", "va"],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
)

# Score: residual RMSE on val (LGBM target) AND TVT RMSE = same thing since target = TVT - anchor
pred_resid_va = model.predict(X_va, num_iteration=model.best_iteration)
rmse_lgbm = float(np.sqrt(((pred_resid_va - y_va) ** 2).mean()))

# CF on val = predict residual = 0
rmse_cf_va = float(np.sqrt((y_va**2).mean()))

# v1-only LGBM on same split for an honest comparison
V1_FEATS = ["dmd", "dz", "gr_roll_mean", "gr_roll_std"]
X_tr_v1, X_va_v1 = tr[V1_FEATS].to_numpy(), va[V1_FEATS].to_numpy()
dtr1 = lgb.Dataset(X_tr_v1, y_tr, feature_name=V1_FEATS)
dva1 = lgb.Dataset(X_va_v1, y_va, feature_name=V1_FEATS, reference=dtr1)
model_v1 = lgb.train(
    params,
    dtr1,
    num_boost_round=500,
    valid_sets=[dtr1, dva1],
    valid_names=["tr", "va"],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
)
pred_v1 = model_v1.predict(X_va_v1, num_iteration=model_v1.best_iteration)
rmse_v1 = float(np.sqrt(((pred_v1 - y_va) ** 2).mean()))

print(f"\n--- Validation RMSE on {len(val_wells)} held-out wells ({len(y_va):,} rows) ---")
print(f"  CF (predict residual=0):  {rmse_cf_va:.2f}")
print(f"  LGBM v1 (4 feats):        {rmse_v1:.2f}   ({rmse_cf_va - rmse_v1:+.2f} vs CF)")
print(
    f"  LGBM v2 (11 feats):       {rmse_lgbm:.2f}   ({rmse_cf_va - rmse_lgbm:+.2f} vs CF, "
    f"{rmse_v1 - rmse_lgbm:+.2f} vs v1)"
)

# Feature importance
imp = pd.DataFrame(
    {
        "feature": FEATURES,
        "gain": model.feature_importance(importance_type="gain"),
        "split": model.feature_importance(importance_type="split"),
    }
).sort_values("gain", ascending=False)
imp["gain_pct"] = (imp["gain"] / imp["gain"].sum() * 100).round(1)
print("\n--- Feature importance (gain) ---")
print(imp.to_string(index=False))

print(f"\nVal wells: {val_wells}")

In [ ]:
from sklearn.model_selection import GroupKFold

ALL_FEATURES = [c for c in fmat.columns if c not in ("well_id", "row_idx", "target_residual")]
V1_FEATS = ["dmd", "dz", "gr_roll_mean", "gr_roll_std"]
V2_FEATS = ALL_FEATURES  # 11 features

groups = fmat["well_id"].to_numpy()
y_all = fmat["target_residual"].to_numpy()

params = dict(
    objective="regression",
    metric="rmse",
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=100,
    feature_fraction=0.9,
    bagging_fraction=0.9,
    bagging_freq=5,
    verbose=-1,
    seed=42,
)


def run_oof(feats_used, label):
    oof = np.full(len(fmat), np.nan)
    fold_meta = []
    feature_imp_total = np.zeros(len(feats_used))
    X_all = fmat[feats_used].to_numpy()

    for fold, (tr_idx, va_idx) in enumerate(
        GroupKFold(n_splits=5).split(X_all, y_all, groups=groups)
    ):
        X_tr, y_tr = X_all[tr_idx], y_all[tr_idx]
        X_va, y_va = X_all[va_idx], y_all[va_idx]
        dtr = lgb.Dataset(X_tr, y_tr, feature_name=feats_used)
        dva = lgb.Dataset(X_va, y_va, feature_name=feats_used, reference=dtr)
        model = lgb.train(
            params,
            dtr,
            num_boost_round=1000,
            valid_sets=[dva],
            valid_names=["va"],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
        )
        pred = model.predict(X_va, num_iteration=model.best_iteration)
        oof[va_idx] = pred
        fold_rmse = float(np.sqrt(((pred - y_va) ** 2).mean()))
        cf_rmse = float(np.sqrt((y_va**2).mean()))
        val_wells = sorted(np.unique(groups[va_idx]))
        fold_meta.append(
            {
                "fold": fold,
                "best_iter": model.best_iteration,
                "fold_rmse": fold_rmse,
                "cf_rmse": cf_rmse,
                "delta": cf_rmse - fold_rmse,
                "n_wells": len(val_wells),
            }
        )
        feature_imp_total += model.feature_importance(importance_type="gain")

    pooled = float(np.sqrt(((oof - y_all) ** 2).mean()))
    pooled_cf = float(np.sqrt((y_all**2).mean()))
    print(f"\n=== {label} ({len(feats_used)} features) ===")
    print(
        f"  Pooled OOF RMSE: {pooled:.3f}   (CF: {pooled_cf:.3f}, delta {pooled_cf - pooled:+.3f})"
    )
    print("  Per fold:")
    for fm in fold_meta:
        print(
            f"    fold {fm['fold']}: best_iter={fm['best_iter']:4d}  "
            f"rmse={fm['fold_rmse']:6.2f}  cf={fm['cf_rmse']:6.2f}  "
            f"delta={fm['delta']:+.2f}  n_wells={fm['n_wells']}"
        )
    return oof, pooled, feature_imp_total


oof_v1, p_v1, _ = run_oof(V1_FEATS, "v1")
oof_v2, p_v2, imp_v2 = run_oof(V2_FEATS, "v2")

print("\n=== Summary ===")
print(f"  v1 OOF RMSE: {p_v1:.3f}")
print(f"  v2 OOF RMSE: {p_v2:.3f}   ({p_v1 - p_v2:+.3f} vs v1)")

imp = pd.DataFrame(
    {
        "feature": V2_FEATS,
        "gain": imp_v2,
    }
).sort_values("gain", ascending=False)
imp["gain_pct"] = (imp["gain"] / imp["gain"].sum() * 100).round(1)
print("\n=== v2 feature importance (summed across folds) ===")
print(imp.to_string(index=False))